In [2]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import numpy as np
import copy
import gc # Для ручной очистки памяти

# ========================== TPU ИМПОРТЫ И НАСТРОЙКИ (НАЧАЛО) ==========================
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.distributed.parallel_loader as pl
    import torch_xla.distributed.xla_multiprocessing as xmp
    from torch_xla.amp import autocast
    TPU_AVAILABLE = True
except ImportError:
    print("⚠️ torch_xla не установлен.")
    TPU_AVAILABLE = False
    # Если TPU недоступен, используем стандартные PyTorch утилиты для CUDA/CPU
    from torch.cuda.amp import autocast, GradScaler

# ========================== ОПТИМИЗИРОВАННЫЕ КОНСТАНТЫ (ЦЕЛЬ) ==========================
if TPU_AVAILABLE:
    try:
        DEVICE = xm.xla_device()
        print(f"✅ Using TPU device: {DEVICE}")
        xm.mark_step()
    except:
        DEVICE = torch.device("cpu")
        TPU_AVAILABLE = False
        print("⚠️ Fallback to CPU")
else:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {DEVICE}")

# 🔧 Параметры для оптимизации памяти и обучения
if TPU_AVAILABLE:
    BATCH_SIZE = 256  # Уменьшенный размер батча для безопасности в XLA
    IMG_SIZE = 256
else:
    BATCH_SIZE = 32 # Стандартный GPU/CPU батч
    IMG_SIZE = 256

GRADIENT_ACCUMULATION_STEPS = 4 if TPU_AVAILABLE else 2
NUM_WORKERS = 2 if TPU_AVAILABLE else 0
TARGET_BALANCE_RATIO = 1.0
print("-" * 50)
print(f"📊 Config: Workers={NUM_WORKERS}, Batch={BATCH_SIZE}, "
      f"Accumulation={GRADIENT_ACCUMULATION_STEPS}, "
      f"Effective Batch={BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")

# ========================== DATASET CLASS (ОСТАВЛЕНО БЕЗ ИЗМЕНЕНИЙ) ==========================
class DeepfakeDataset(Dataset):
    def __init__(self, img_dir, ids_list, labels=None, transform=None):
        self.img_dir = img_dir
        self.labels = labels
        self.transform = transform
        self.files_ids = ids_list

    def __len__(self):
        return len(self.files_ids)

    def __getitem__(self, idx):
        img_id = self.files_ids[idx]
        fname = f"{img_id}.jpg"
        img_path = os.path.join(self.img_dir, fname)
        if not os.path.exists(img_path):
            # Возвращаем нуль-тензор и некорректный лейбл в случае ошибки
            return torch.zeros(3, IMG_SIZE, IMG_SIZE), -1 
        try:
            img = Image.open(img_path).convert("RGB")
            if self.transform:
                img = self.transform(img)
            label = self.labels.get(img_id, -1)
            return img, label
        except Exception:
             return torch.zeros(3, IMG_SIZE, IMG_SIZE), -1

def collate_fn(batch):
    # Фильтрация батчей с ошибками загрузки (-1 лейбл)
    valid_batch = [(img, lbl) for img, lbl in batch if lbl != -1]
    if not valid_batch:
        return torch.zeros(1, 3, IMG_SIZE, IMG_SIZE), torch.zeros(1, dtype=torch.long)
    return torch.utils.data.dataloader.default_collate(valid_batch)

# ========================== TRANSFORMS (УСИЛЕННАЯ АУГМЕНТАЦИЯ) ==========================
mean = torch.tensor([0.5192, 0.4276, 0.3844])
std = torch.tensor([0.2860, 0.2640, 0.2635])

transform_rule = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    # Дополнительная робастность: имитация разных углов съемки
    transforms.RandomRotation(degrees=15), 
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    # Симуляция изменения освещения (оттенки серого)
    transforms.RandomGrayscale(p=0.3), 
    transforms.ToTensor(),
    transforms.Normalize(mean=mean.tolist(), std=std.tolist())
])

# ========================== LOAD DATA & OVERSAMPLING (БЕЗ ИЗМЕНЕНИЙ) ==========================
train_dir = "dataset/train_images"
csv_dir = "dataset/train_solution.csv"

# 1. Загрузка меток (Пропущена детальная реализация для краткости, предполагается работа)
labels = {}
if os.path.exists(csv_dir):
    with open(csv_dir, "r") as file:
        for line in file:
            line = line.strip()
            if "," in line:
                idx, val = line.split(",")
                if idx.isdigit() and val.isdigit():
                    labels[int(idx)] = int(val)

all_ids = list(labels.keys())
all_labels_stratify = [labels[i] for i in all_ids]

# 2. Разделение на train/val (стратификация сохраняется)
train_ids_unique, val_ids = train_test_split(
    all_ids, test_size=0.2, random_state=42, stratify=all_labels_stratify
)

# 3. Расчет весов
count_real = sum(1 for i in all_ids if labels[i] == 0)
count_fake = sum(1 for i in all_ids if labels[i] == 1)
total_samples = len(all_ids)

# Обновленный расчет весов: нормализованные веса
weight_real = total_samples / (2 * max(count_real, 1))
weight_fake = total_samples / (2 * max(count_fake, 1))
class_weights = torch.tensor([weight_real, weight_fake]).float()

# Oversampling logic (Сохранена)
real_count_train = sum(1 for i in train_ids_unique if labels[i] == 0)
fake_count_train = sum(1 for i in train_ids_unique if labels[i] == 1)
balanced_train_ids = list(train_ids_unique)

if fake_count_train > 0:
    target_fake_count = int(real_count_train * TARGET_BALANCE_RATIO)
    oversample_multiplier = min(target_fake_count // fake_count_train, 2)
    
    fake_ids_to_oversample = [id for id in train_ids_unique if labels[id] == 1]
    for _ in range(oversample_multiplier - 1):
        balanced_train_ids.extend(fake_ids_to_oversample)
    
    remainder = min(target_fake_count % fake_count_train, len(fake_ids_to_oversample))
    if remainder > 0:
        # Более безопасный способ выборки остатка
        temp_pool = list(fake_ids_to_oversample) + [i for i in fake_ids_to_oversample] * 2
        remainder_selection = np.random.choice(temp_pool, remainder, replace=True)
        balanced_train_ids.extend(remainder_selection)

print(f"\n✅ Final Balanced train size: {len(balanced_train_ids)}")


# ========================== DATASETS & DATALOADERS (ФИНАЛИЗАЦИЯ) ==========================
train_dataset = DeepfakeDataset(train_dir, ids_list=balanced_train_ids, labels=labels, transform=transform_rule)
val_dataset = DeepfakeDataset(train_dir, ids_list=val_ids, labels=labels, transform=transform_rule)

# Создание DataLoader
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=False, 
    drop_last=True, collate_fn=collate_fn, prefetch_factor=2 if NUM_WORKERS > 0 else None
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=False, 
    collate_fn=collate_fn, prefetch_factor=2 if NUM_WORKERS > 0 else None
)

# Перевод в специальный DataLoader для TPU (MpDeviceLoader)
if TPU_AVAILABLE:
    train_loader = pl.MpDeviceLoader(train_loader, DEVICE)
    val_loader = pl.MpDeviceLoader(val_loader, DEVICE)


# ========================== МОДЕЛЬ ARCHITECTURE V2.0 (RESNET-BASED) ==========================

class ResidualBlock(nn.Module):
    """Мощный остаточный блок ResNet-подобного дизайна."""
    def __init__(self, in_c, out_c):
        super().__init__()
        # Ветвь 1: Conv -> BN -> ReLU
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
        # Ветвь 2: Conv -> BN (без ReLU на выходе для остаточного соединения)
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c)
        )
        # Shortcut (Связь): Линейное преобразование каналов, если необходимо
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_c, out_c, 1, bias=False),
            nn.BatchNorm2d(out_c)
        ) if in_c != out_c else nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.conv1(x)
        out = self.conv2(out)
        # Остаточное соединение: ReLU(Output + Identity)
        return torch.relu(out + identity)

class DeepfakeCNN(nn.Module):
    """Архитектура, основанная на Residual Blocks."""
    def __init__(self):
        super().__init__()
        base_channels = 32 if TPU_AVAILABLE else 64
        
        # Начальный слой: stride=2 уменьшает размерность (H/W) и добавляет каналы.
        self.initial_conv = nn.Sequential(
            nn.Conv2d(3, base_channels, kernel_size=3, padding=1, stride=2, bias=False),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(inplace=True)
        )
        # Прогрессия каналов: 64 -> 128 -> 256 -> 512
        self.layer1 = ResidualBlock(base_channels, base_channels * 2) 
        self.layer2 = ResidualBlock(base_channels * 2, base_channels * 4) 
        self.layer3 = ResidualBlock(base_channels * 4, base_channels * 8) 
        # Финальный блок - для стабилизации и последнего извлечения признаков
        self.final_conv = nn.Sequential(
            nn.Conv2d(base_channels * 8, base_channels * 8, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_channels * 8)
        )

        self.pool = nn.MaxPool2d(2) # Уменьшение H/W на 2 после каждого блока
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        # Последний блок: base_channels * 8 (512) каналов
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(base_channels * 8, 2)
        )

    def forward(self, x):
        x = self.initial_conv(x)
        # Проход через блоки с понижением размерности H/W на каждом этапе
        x = self.pool(self.layer1(x))
        x = self.pool(self.layer2(x))
        x = self.pool(self.layer3(x))
        x = self.pool(self.final_conv(x)) 
        
        # Финальная обработка и классификация
        x = self.global_pool(x)
        return self.classifier(x)

# ========================== ФУНКЦИЯ ОБУЧЕНИЯ (СОХРАНЕНА С ЛОГИКОЙ АКТИВАЦИИ) ==========================
def train_model(model, train_loader, val_loader, class_weights, num_epochs=70):
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
    # Снижаем Learning Rate и добавляем Weight Decay для стабильности
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5) 
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    
    scaler = GradScaler() if not TPU_AVAILABLE and torch.cuda.is_available() else None
    
    best_f1 = 0.0
    patience_counter = 0
    best_model_state = None
    MODEL_SAVE_DIR = "/kaggle/working" if TPU_AVAILABLE else "models"
    os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

    print("\n" + "="*50)
    print("🚀 START TRAINING LOOP")
    print("="*50)


    for epoch in range(num_epochs):
        # ==================== TRAIN PHASE ====================
        model.train()
        train_loss = 0.0
        train_preds, train_true = [], []
        
        if not TPU_AVAILABLE:
            iterator = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
        else:
            iterator = train_loader # XLA handles iteration internally

        # (Внимание: В этом блоке должна быть полная логика градиентного накопления. 
        # Я сохраняю структуру, предполагая идеальную работоспособность оригинальной логики.)
        
        for step, (images, labels) in enumerate(iterator):
            if images.numel() == 0: continue

            # Перевод данных на устройство
            if not TPU_AVAILABLE and torch.cuda.is_available():
                images = images.to(DEVICE)
                labels = labels.to(DEVICE)
            elif TPU_AVAILABLE:
                 # В XLA, данные уже могут быть на DEVICE от MpDeviceLoader
                 pass 

            # Forward pass & Loss calculation (использование autocast и scaler остается ключевым!)
            if TPU_AVAILABLE:
                with autocast(DEVICE, dtype=torch.bfloat16):
                    outputs = model(images)
                    loss = criterion(outputs, labels) / GRADIENT_ACCUMULATION_STEPS
                loss.backward()
            else: # GPU/CPU Path
                if scaler:
                    with autocast():
                        outputs = model(images)
                        loss = criterion(outputs, labels) / GRADIENT_ACCUMULATION_STEPS
                    scaler.scale(loss).backward()
                else:
                    outputs = model(images)
                    loss = criterion(outputs, labels) / GRADIENT_ACCUMULATION_STEPS
                    loss.backward()
            
            # Gradient accumulation step (логика остается без изменений)
            if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                if TPU_AVAILABLE:
                    xm.optimizer_step(optimizer, barrier=True)
                else: # GPU/CPU path
                    if scaler:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                        optimizer.step()
                
                optimizer.zero_grad()
                if TPU_AVAILABLE:
                    xm.mark_step()

            # Сбор статистики (должен быть реализован для полной версии)
            train_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS 
            preds = outputs.argmax(dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_true.extend(labels.cpu().numpy())


        # ==================== VALIDATION PHASE ====================
        model.eval()
        val_loss = 0.0
        val_preds, val_true = [], []
        
        with torch.no_grad():
            for images, labels in val_loader:
                if images.numel() == 0: continue

                # (Обработка устройства для валидации)
                if not TPU_AVAILABLE and torch.cuda.is_available():
                    images = images.to(DEVICE)
                    labels = labels.to(DEVICE)
                
                # Forward pass & Loss calculation
                if TPU_AVAILABLE:
                    with autocast(DEVICE, dtype=torch.bfloat16):
                        outputs = model(images)
                        loss = criterion(outputs, labels)
                else:
                    if scaler:
                        with autocast():
                            outputs = model(images)
                            loss = criterion(outputs, labels)
                    else:
                        outputs = model(images)
                        loss = criterion(outputs, labels)

                val_loss += loss.item()
                preds = outputs.argmax(dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_true.extend(labels.cpu().numpy())


        # ==================== METRICS & CHECKPOINTING ====================
        avg_train_loss = train_loss / max(len(train_loader), 1) # Упрощенная метрика, так как полная логика накопления усложнит расчет
        avg_val_loss = val_loss / max(len(val_loader), 1)

        try:
            # Используем zero_division=0 для устойчивости.
            train_f1 = f1_score(train_true, train_preds, average='binary', zero_division=0)
            val_f1 = f1_score(val_true, val_preds, average='binary', zero_division=0)
        except ValueError:
             train_f1, val_f1 = 0.0, 0.0

        val_acc = accuracy_score(val_true, val_preds)
        
        log_msg = (f"Epoch {epoch+1:2d}/{num_epochs} | "
                   f"Train Loss: {avg_train_loss:.4f}, F1: {train_f1:.4f} | "
                   f"Val Loss: {avg_val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")

        if TPU_AVAILABLE:
            xm.master_print(log_msg)
        else:
            print(log_msg)
        
        # Сохранение лучшей модели
        if val_f1 > best_f1:
            best_f1 = val_f1
            patience_counter = 0
            best_model_state = copy.deepcopy(model.state_dict())
            save_path = f"{MODEL_SAVE_DIR}/best_model_f1_{val_f1:.4f}.pth"
            
            if TPU_AVAILABLE:
                xm.save(best_model_state, save_path)
                xm.master_print(f"✅ Saved: {save_path}")
            else:
                torch.save(best_model_state, save_path)
                print(f"✅ Saved: {save_path}")
        else:
            patience_counter += 1
            if patience_counter >= 7:
                break
        
        scheduler.step()
        gc.collect() # Обязательная чистка памяти

    # Загрузка лучшей модели перед возвратом
    if best_model_state:
        model.load_state_dict(best_model_state)
    return model


# ========================== ФУНКЦИЯ ЗАПУСКА (CORE EXECUTION) ==========================
def run_training():
    global DEVICE # Убеждаемся, что все используют глобально определенное устройство
    model = DeepfakeCNN().to(DEVICE)
    total_params = sum(p.numel() for p in model.parameters())
    print("\n" + "="*50)
    print(f"✅ Model initialized on {DEVICE}. Parameters: {total_params:,}")
    print("🚀 Starting Deepfake Detection Training...")
    print("="*50)

    # Передача рассчитанных весов и запуска обучения
    trained_model = train_model(
        model, 
        train_loader, 
        val_loader, 
        class_weights, # Взвешенные потери
        num_epochs=70  # Увеличиваем эпохи для лучшей конвергенции
    )

    if not TPU_AVAILABLE:
        torch.save(trained_model.state_dict(), "final_deepfake_cnn_v2_optimized.pth")
        print("\n✅ Final model saved successfully!")


# ========================== MAIN EXECUTION BLOCK (Исполнение) ==========================
if __name__ == '__main__':
    run_training()



C:\Users\Иван\AppData\Local\Temp\ipykernel_15592\2528923035.py:263: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if not TPU_AVAILABLE and torch.cuda.is_available() else None


⚠️ torch_xla не установлен.
Using device: cuda
--------------------------------------------------
📊 Config: Workers=0, Batch=32, Accumulation=2, Effective Batch=64

✅ Final Balanced train size: 52800

✅ Model initialized on cuda. Parameters: 7,185,474
🚀 Starting Deepfake Detection Training...

🚀 START TRAINING LOOP


Epoch 1/70 [Train]:   0%|          | 0/1650 [00:00<?, ?it/s]C:\Users\Иван\AppData\Local\Temp\ipykernel_15592\2528923035.py:309: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 1/70 [Train]:  40%|███▉      | 653/1650 [12:30<19:05,  1.15s/it]


KeyboardInterrupt: 